# 13 · Explore: profiles, masks and composites in one-liners

A playground. Every cell is short and stands on the CONFIG cell only, so copy, change and recombine freely.

| What | Call | Gives |
|---|---|---|
| SAM's own hourly STAT profiles (instant) | `S = stat_profiles(EXPERIMENT, RV_LIST)` | Dataset `(rv, z)`: `MSE`, `MSECLD` (cloud MSE), `CLD`, `TABS`, `QV`, `RELH`, … (MSE in **K**: ×1.004 → kJ/kg) |
| Composites with any mask (3D output; minutes first time, then cached) | `C = load(EXPERIMENT, RV_LIST, masks.cloud)` | Dataset `(rv, group, z)`: every 3D field + `MSE`, `MSE_frozen`, `THETA_E` and saturated versions; coords `RH`, `fraction`, `fraction_z` |
| One panel | `plot_profiles(C, "MSE", group="cloud")` / `plot_profiles(C, "MSE", by="group", rv=0)` | Lines per r_v (coloured by RH) or per group; `anomaly=True`, `xlim=`, `zmax=`, `sat=False` |
| Many panels | `plot_grid(C, ["W", "QN", "MSE"], group="cloud")` | Same options |
| Stability | `stability_profiles(C)` | `N2`, `N2_m`, `N2_m_entropy`, `N2_m_loading` (Emanuel 6.1.7 / 6.2.10) |

Masks come in two kinds: **column** masks (the same columns at every height, e.g. near-surface W) and **level** masks (chosen at each height, e.g. *cloudy here*). The library ones are in `samheat/analysis/masks.py`; write your own in section 7.

In [ ]:
# RUN THIS CELL FIRST: lets this notebook find the samheat code in this repo (no install needed)
import os, sys, pathlib
def _find_repo():
    starts = [pathlib.Path(os.environ.get("JPY_SESSION_NAME", ".")).resolve(), pathlib.Path.cwd().resolve()]
    for s in starts:
        for d in [s, *s.parents]:
            if (d / "samheat" / "__init__.py").exists():
                return d
    for d in [pathlib.Path("/scratch/ttg2015/Torch/ExtremeHeat/SAM_HEAT"), pathlib.Path.home() / "SAM_HEAT"]:
        if (d / "samheat" / "__init__.py").exists():
            return d
    raise FileNotFoundError("Could not find the SAM_HEAT folder. Set REPO by hand, e.g. REPO = pathlib.Path('/scratch/.../SAM_HEAT')")
REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import samheat
print("using samheat from", pathlib.Path(samheat.__file__).parent)

In [ ]:
# ---- CONFIG ----
import matplotlib.pyplot as plt
from samheat.paths import SITE
from samheat.analysis.explore import load, stat_profiles, plot_profiles, plot_grid, masks
from samheat.analysis.stability import stability_profiles

EXPERIMENT = "vdd2025_vddgrid"          # or "vdd2025_repro", "vdd2025_vddgrid_U5", ...
RV_LIST = [0, 200, 500, 1000, 2000]
TMIN, TMAX = 30, 60                     # days of Stage 2

In [ ]:
# ---- What masks exist? ----
for kind, group in (("column", masks.COLUMN_MASKS), ("level", masks.LEVEL_MASKS)):
    for name, fn in group.items():
        print(f"{kind:6s}  masks.{name:15s} {fn.__doc__.strip().splitlines()[0]}")

## 1. SAM's own STAT profiles (no 3D reading: instant)
`MSECLD` is SAM's cloud-conditional MSE (cloud = liquid + ice > min(0.01 g/kg, 1 % of saturation)), written every hour. SAM writes MSE in K (h/c_p).

In [ ]:
S = stat_profiles(EXPERIMENT, RV_LIST, tmin=TMIN, tmax=TMAX)
S["MSE_kJ"] = 1.004 * S.MSE                 # K -> kJ/kg
S["MSECLD_kJ"] = 1.004 * S.MSECLD.where(S.CLD > 0)   # cloud MSE only where there is cloud
print([v for v in S.data_vars][:40], "...")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
plot_profiles(S, "MSE_kJ", ax=ax[0], xlim=(320, 350))
plot_profiles(S, "MSECLD_kJ", ax=ax[1], xlim=(320, 350), legend=False)
plot_profiles(S, "CLD", ax=ax[2], legend=False)
plt.show()

## 2. Composites from the 3D output
The first call per mask reads every snapshot (minutes per run); after that it loads from the cache in seconds.

In [ ]:
C_cloud = load(EXPERIMENT, RV_LIST, masks.cloud, tmin=TMIN, tmax=TMAX)                  # level mask: cloudy vs clear
C_cloud

In [ ]:
C_w = load(EXPERIMENT, RV_LIST, masks.symmetric, k=1, X=0.3, tmin=TMIN, tmax=TMAX)      # column mask: near-surface W

## 3. Quick plots

In [ ]:
plot_profiles(C_cloud, "MSE", group="cloud"); plt.show()              # cloud MSE per r_v (dashed: saturated MSE*)

In [ ]:
plot_profiles(C_cloud, "MSE", by="group", rv=RV_LIST[0]); plt.show()  # cloud vs clear vs domain mean, one run

In [ ]:
plot_grid(C_cloud, ["QN", "W", "TABS", "QV", "MSE", "MSE_frozen", "THETA_E", "QP"], group="cloud"); plt.show()

In [ ]:
plot_profiles(C_cloud, "TABS", group="cloud", anomaly=True); plt.show()   # cloud minus domain mean

In [ ]:
plot_profiles(C_cloud, "fraction_z", group="cloud", xlim="auto"); plt.show()   # fraction of points that are cloudy, per level

## 4. Check: our cloud composite vs SAM's own `MSECLD`
They should be close below the freezing level, which is a good test of the 3D pipeline. Differences: SAM averages hourly, we use 3-hourly snapshots, and SAM's MSE has no ice term (compare with `MSE_frozen` below ~5 km).

In [ ]:
from samheat.analysis.explore import rh_colors
col = rh_colors(C_cloud)
fig, ax = plt.subplots(figsize=(5, 5))
for rv in C_cloud.rv.values:
    ax.plot(C_cloud.MSE_frozen.sel(rv=rv, group="cloud"), C_cloud.z / 1000, c=col[rv], label=f"rv {rv:g} (3D composite)")
    ax.plot(S.MSECLD_kJ.sel(rv=rv), S.z / 1000, c=col[rv], ls=":", label=f"rv {rv:g} (STAT MSECLD)")
ax.set_xlim(320, 350); ax.set_ylim(0, 16); ax.set_xlabel("cloud MSE (kJ/kg)"); ax.set_ylabel("z (km)"); ax.legend(fontsize=7)
plt.show()

## 5. Stability of each group (Emanuel 6.1.7 / 6.2.10)

In [ ]:
ST = stability_profiles(C_cloud)
plot_grid(ST, ["N2", "N2_m", "N2_m_entropy", "N2_m_loading"], group="clear"); plt.show()

## 6. Do it yourself with xarray
Everything above is a plain xarray Dataset: `.sel`, arithmetic, `.mean`, `.plot` all work.

In [ ]:
diff = C_cloud.MSE.sel(group="cloud") - C_cloud.MSE.sel(group="clear")      # (rv, z): cloud minus clear-air MSE
diff.plot(y="z", hue="rv"); plt.ylim(0, 16000); plt.axvline(0, c="k", lw=.5); plt.show()
diff.sel(z=slice(1000, 5000)).mean("z").to_pandas()                          # one number per run

## 7. Your own mask
A mask gets one snapshot `f` (dict of 3D arrays `(z, y, x)` in SAM units, plus the derived fields) and the heights `z`. Return `{group: boolean array}`: `(y, x)` for columns or `(z, y, x)` for level by level. Helpers you can use: `masks.w_at_levels(f)`, `masks.buoyancy(f)`, `masks.parcel_cape(f, z)`, `masks.sam_cloud_threshold(f, z)`.

In [ ]:
def my_mask(f, z, w_min=2.0):
    """Strong cloudy updrafts at each level vs everything else."""
    strong = (f["QN"] > masks.sam_cloud_threshold(f, z)) & (masks.w_at_levels(f) > w_min)
    return dict(strong=strong, rest=~strong)

C_my = load(EXPERIMENT, RV_LIST, my_mask, w_min=2.0, tmin=TMIN, tmax=TMAX)
plot_profiles(C_my, "MSE", group="strong"); plt.show()